# Validação GPU — JAX Unified JIT + Sprint 12 vmap_real (v1.5.0 → v1.6.0)

**Projeto:** Geosteering AI v2.0 — Simulador Python  
**Sprint:** 10 Phase 2 (baseline) + Sprint 12 E11 (regressão + vmap_real)  
**Versão:** v1.5.0 (PR #24) → v1.6.0 (PR #25 — Sprint 12)  
**Autor:** Daniel Leal  

## Objetivo

Validar gates Sprint 10 (baseline XLA consolidation) e Sprint 12 (regressão 600×3 + vmap_real):

### Sprint 10 — Gates Baseline (Seções 1–10)
| Gate | Métrica | Target | Status |
|:-----|:--------|:------:|:------:|
| **G1** | XLA programs oklahoma_28 unified | == 1 | ⏳ |
| **G2** | VRAM peak unified oklahoma_28 | < 1 GB | ⏳ |
| **G3** | Throughput unified vs bucketed GPU | ≥ 3× | ⏳ |
| **G4** | Paridade numérica `\|H_b - H_u\|` | < 1e-10 | ⏳ |
| **G5** | jacfwd unified oklahoma_28 alta-ρ | 0 NaN/Inf | ⏳ |

### Sprint 12 — Gates E11 (Seções 11–14)
| Gate | Métrica | Target | Status |
|:-----|:--------|:------:|:------:|
| **G6** | find_layers_tr_jax paridade vs Numba | diff == 0 | ⏳ |
| **G7** | vmap_real paridade vs Python loop | < 1e-12 | ⏳ |
| **G8** | vmap_real speedup em multi-dip T4 | ≥ 1.5× | ⏳ |
| **G9** | Regressão 600×3 diagnosticada | root cause id. | ⏳ |

## Instruções

1. `Ambiente de execução → Alterar tipo de ambiente de execução → GPU T4` (ou A100)
2. Execute todas as células em sequência (`Ctrl+F9` ou `Ambiente de execução → Executar tudo`)
3. **Seções 1–10**: validação Sprint 10 baseline (XLA consolidation, VRAM, throughput)
4. **Seções 11–14**: Sprint 12 — diagnóstico regressão 600×3 + find_layers_tr_jax + vmap_real
5. Relatório final Sprint 10 → Seção 10 | Relatório final Sprint 12 → **Seção 14**

---
## Seção 1 — Detecção de Ambiente e Instalação

In [ ]:
import subprocess
import sys
import os

# ── Detecta GPU via nvidia-smi ────────────────────────────────────────────────
try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total,driver_version',
         '--format=csv,noheader'],
        capture_output=True, text=True, timeout=10
    )
    HAS_GPU = result.returncode == 0 and result.stdout.strip()
    GPU_INFO = result.stdout.strip() if HAS_GPU else 'N/A'
except (FileNotFoundError, subprocess.TimeoutExpired):
    HAS_GPU = False
    GPU_INFO = 'N/A'

# ── Detecta A100 para benchmark mais intenso ──────────────────────────────────
IS_A100 = HAS_GPU and 'A100' in GPU_INFO
IS_T4   = HAS_GPU and 'T4'   in GPU_INFO

print(f'╔══════════════════════════════════════════════════════╗')
print(f'║  Ambiente de Execução                               ║')
print(f'╠══════════════════════════════════════════════════════╣')
print(f'║  GPU detectada : {HAS_GPU}                                ║'.replace('True ', 'True  '))
print(f'║  GPU info      : {GPU_INFO[:40]:40s}  ║')
print(f'║  Python        : {sys.version.split()[0]:10s}                          ║')
print(f'╚══════════════════════════════════════════════════════╝')

if not HAS_GPU:
    print('\n⚠️  AVISO: GPU não detectada! Altere o runtime para GPU.')
    print('   Vá em: Ambiente de execução → Alterar tipo → GPU T4')
else:
    print(f'\n✅ GPU pronta para validação.')

In [ ]:
# ── Instalação de dependências ────────────────────────────────────────────────
# JAX com CUDA 12 se GPU disponível, jax[cpu] caso contrário
if HAS_GPU:
    print('Instalando jax[cuda12]...')
    !pip install -q 'jax[cuda12]' numba
else:
    print('Instalando jax[cpu] (sem GPU)...')
    !pip install -q 'jax[cpu]' numba

print('Instalação de dependências concluída.')

In [ ]:
# ── Clone do repositório e instalação do pacote ───────────────────────────────
import os

REPO_DIR = '/content/geosteering-ai'

if not os.path.exists(REPO_DIR):
    print('Clonando repositório...')
    # Nota: se o repositório for privado, use um token de acesso pessoal:
    # !git clone -q https://<TOKEN>@github.com/daniel-guitarplayer-8/geosteering-ai.git {REPO_DIR}
    !git clone -q https://github.com/daniel-guitarplayer-8/geosteering-ai.git {REPO_DIR}
else:
    print('Repositório já clonado. Atualizando...')
    !cd {REPO_DIR} && git pull -q origin main

os.chdir(REPO_DIR)
!pip install -q -e . 2>&1 | tail -5

# Verifica versão instalada
from geosteering_ai.simulation import __version__ as SIM_VERSION
print(f'\n✅ geosteering_ai.simulation versão: {SIM_VERSION}')
assert SIM_VERSION >= '1.5.0', f'Versão esperada ≥ 1.5.0, encontrado: {SIM_VERSION}'

---
## Seção 2 — Verificação do Ambiente JAX

In [ ]:
import jax
import jax.numpy as jnp
import numpy as np

# Habilita float64 — CRÍTICO para paridade com Numba
jax.config.update('jax_enable_x64', True)

from geosteering_ai.simulation._jax import HAS_JAX
from geosteering_ai.simulation._numba.propagation import HAS_NUMBA

print('╔══════════════════════════════════════════════════════╗')
print('║  Ambiente JAX                                       ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  JAX versão    : {jax.__version__:10s}                       ║')
print(f'║  Backend padrão: {jax.default_backend():10s}                       ║')
print(f'║  Devices       : {str(jax.devices())[:35]:35s}  ║')
print(f'║  HAS_JAX       : {str(HAS_JAX):5s}                                 ║')
print(f'║  HAS_NUMBA     : {str(HAS_NUMBA):5s}                                 ║')
print(f'║  NumPy versão  : {np.__version__:10s}                       ║')
print('╚══════════════════════════════════════════════════════╝')

BACKEND = jax.default_backend()
IS_GPU_BACKEND = BACKEND in ('gpu', 'cuda')

if IS_GPU_BACKEND:
    print(f'\n✅ JAX rodando em GPU ({BACKEND}).')
else:
    print(f'\n⚠️  JAX rodando em CPU. Gates G2 e G3 serão medidos em CPU.')
    print('   Altere o runtime para GPU para validação completa.')

assert HAS_JAX, 'JAX não está disponível!'

In [ ]:
# ── Smoke test JAX: operação simples para confirmar backend ───────────────────
x = jnp.ones((100, 100), dtype=jnp.float64)
y = jnp.dot(x, x)
y.block_until_ready()
print(f'Smoke test JAX: dot((100,100) × (100,100)) = {float(y[0,0]):.1f} ✅')
print(f'dtype float64 ativo: {y.dtype} ✅')

---
## Seção 3 — Gate G1: Consolidação XLA (44 → 1 programa)

In [ ]:
from geosteering_ai.simulation._jax import (
    clear_unified_jit_cache,
    count_compiled_xla_programs,
)
from geosteering_ai.simulation._jax.forward_pure import (
    build_static_context,
    clear_jit_cache,
    forward_pure_jax,
)
from geosteering_ai.simulation.validation.canonical_models import get_canonical_model

def _clear_all_caches():
    """Limpa caches JAX para isolar contagens XLA entre testes."""
    clear_jit_cache()
    clear_unified_jit_cache()


def _build_ctx(model_name, strategy, n_pos=100):
    """Constrói contexto estático para um modelo canônico."""
    m = get_canonical_model(model_name)
    z = np.linspace(m.min_depth - 2, m.max_depth + 2, n_pos)
    return build_static_context(
        m.rho_h, m.rho_v, m.esp, z,
        freqs_hz=np.array([20000.0]),
        tr_spacing_m=1.0, dip_deg=0.0,
        strategy=strategy,
    ), m


print('Gate G1: Verificando consolidação XLA em oklahoma_28...')
print('(oklahoma_28 tem 28 camadas → 44 buckets com strategy=bucketed)\n')

results_xla = {}
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    for strategy in ['bucketed', 'unified']:
        _clear_all_caches()
        ctx, m = _build_ctx(model_name, strategy, n_pos=100)
        # Warmup (compila o JIT)
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
        H.block_until_ready()
        xla_count = count_compiled_xla_programs(ctx)
        results_xla[(model_name, strategy)] = xla_count

print(f'{'Modelo':15s} {'Camadas':>8s} {'XLA bucketed':>14s} {'XLA unified':>13s} {'Consolidação':>14s}')
print('-' * 68)
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    m = get_canonical_model(model_name)
    xla_b = results_xla[(model_name, 'bucketed')]
    xla_u = results_xla[(model_name, 'unified')]
    consolidacao = f'{xla_b}×' if xla_u == 1 else f'{xla_b/xla_u:.1f}×'
    status = '✅' if xla_u == 1 else '❌'
    print(f'{model_name:15s} {len(m.rho_h):>8d} {xla_b:>14d} {xla_u:>13d} {consolidacao:>10s} {status}')

# Gate G1
G1_PASS = results_xla[('oklahoma_28', 'unified')] == 1
print(f'\nGate G1: XLA unified oklahoma_28 == 1: {"✅ PASS" if G1_PASS else "❌ FAIL"}')

---
## Seção 4 — Gate G4: Paridade Numérica (unified vs bucketed)

In [ ]:
print('Gate G4: Verificando paridade numérica unified vs bucketed...')
print('Gate: max|H_b - H_u| < 1e-10\n')

results_parity = {}
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    _clear_all_caches()
    ctx_b, m = _build_ctx(model_name, 'bucketed', n_pos=60)
    H_b = forward_pure_jax(ctx_b.rho_h_jnp, ctx_b.rho_v_jnp, ctx_b)
    H_b.block_until_ready()

    _clear_all_caches()
    ctx_u, _ = _build_ctx(model_name, 'unified', n_pos=60)
    H_u = forward_pure_jax(ctx_u.rho_h_jnp, ctx_u.rho_v_jnp, ctx_u)
    H_u.block_until_ready()

    max_abs = float(jnp.abs(H_b - H_u).max())
    results_parity[model_name] = max_abs

print(f'{'Modelo':15s} {'max|H_b - H_u|':>16s} {'Gate 1e-10':>12s} {'Status':>8s}')
print('-' * 56)
for model_name, max_abs in results_parity.items():
    ok = max_abs < 1e-10
    print(f'{model_name:15s} {max_abs:>16.3e} {'<1e-10':>12s} {'✅ PASS' if ok else '❌ FAIL':>8s}')

G4_PASS = all(v < 1e-10 for v in results_parity.values())
print(f'\nGate G4: Paridade <1e-10 em todos os modelos: {"✅ PASS" if G4_PASS else "❌ FAIL"}')

---
## Seção 5 — Gate G5: jacfwd Diferenciável em Alta Resistividade

In [ ]:
print('Gate G5: jacfwd unified em oklahoma_28 com ρ_high ≈ 1500 Ω·m...')
print('(Alta resistividade é cenário crítico para NaN/Inf em sensores LWD)\n')

_clear_all_caches()
m_28 = get_canonical_model('oklahoma_28')

# Escalonar resistividades para ρ > 1000 Ω·m (cenário duro)
rho_h_high = np.clip(m_28.rho_h, a_min=1.0, a_max=None) * 15.0
rho_v_high = np.clip(m_28.rho_v, a_min=1.0, a_max=None) * 15.0
z_short = np.linspace(m_28.min_depth, m_28.max_depth, 20)

ctx_jac = build_static_context(
    rho_h_high, rho_v_high, m_28.esp, z_short,
    freqs_hz=np.array([20000.0]),
    tr_spacing_m=1.0, dip_deg=0.0,
    strategy='unified',
)

def _fwd_fn(rh, rv):
    return forward_pure_jax(rh, rv, ctx_jac)

# jacfwd: diferenciação automática sobre rho_h
J = jax.jacfwd(_fwd_fn, argnums=0)(ctx_jac.rho_h_jnp, ctx_jac.rho_v_jnp)
J.block_until_ready()

n_nan = int(jnp.isnan(J).sum())
n_inf = int(jnp.isinf(J).sum())
n_total = J.size

print(f'Jacobiano shape: {J.shape}')
print(f'NaNs: {n_nan}/{n_total}')
print(f'Infs: {n_inf}/{n_total}')
print(f'Max |J|: {float(jnp.abs(J).max()):.4e}')

G5_PASS = (n_nan == 0) and (n_inf == 0)
print(f'\nGate G5: jacfwd unified alta-ρ (0 NaN, 0 Inf): {"✅ PASS" if G5_PASS else "❌ FAIL"}')

---
## Seção 6 — Gate G2: VRAM Peak (target < 1 GB para oklahoma_28)

In [ ]:
import gc

def _get_gpu_vram_mb():
    """Retorna VRAM usada em MB via nvidia-smi. Retorna None se não disponível."""
    try:
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            return float(result.stdout.strip().split('\n')[0])
    except Exception:
        pass
    return None


def _measure_vram_for_strategy(model_name, strategy, n_pos=100):
    """Mede VRAM peak após warm-up para um modelo + estratégia."""
    gc.collect()
    _clear_all_caches()

    vram_antes = _get_gpu_vram_mb() or 0.0

    ctx, _ = _build_ctx(model_name, strategy, n_pos=n_pos)
    H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()

    vram_depois = _get_gpu_vram_mb() or 0.0
    delta_mb = vram_depois - vram_antes

    return vram_antes, vram_depois, delta_mb


print('Gate G2: Medindo VRAM peak (oklahoma_28, 100 posições)...')
print('Target: VRAM unified < 1000 MB (≈ 250 MB esperado vs ≈ 11 GB bucketed)\n')

if not HAS_GPU:
    print('⚠️  GPU não disponível — Gate G2 não pode ser medido em CPU.')
    print('   Execute este notebook em runtime GPU T4 para validação completa.')
    G2_PASS = None  # Inconclusivo
else:
    vram_results = {}
    for strategy in ['bucketed', 'unified']:
        vram_antes, vram_depois, delta = _measure_vram_for_strategy(
            'oklahoma_28', strategy, n_pos=100
        )
        vram_results[strategy] = {
            'antes_mb': vram_antes,
            'depois_mb': vram_depois,
            'delta_mb': delta,
        }

    print(f'{'Estratégia':12s} {'VRAM antes (MB)':>17s} {'VRAM depois (MB)':>18s} {'Delta (MB)':>12s}')
    print('-' * 64)
    for s, r in vram_results.items():
        print(f'{s:12s} {r["antes_mb"]:>17.1f} {r["depois_mb"]:>18.1f} {r["delta_mb"]:>12.1f}')

    delta_u = vram_results['unified']['delta_mb']
    delta_b = vram_results['bucketed']['delta_mb']
    reducao_x = delta_b / delta_u if delta_u > 0 else float('inf')

    print(f'\nRedução VRAM: {reducao_x:.1f}× (unified vs bucketed)')

    G2_PASS = delta_u < 1000  # Gate: < 1 GB
    print(f'Gate G2: VRAM unified oklahoma_28 < 1 GB: {"✅ PASS" if G2_PASS else "❌ FAIL"}')
    print(f'  VRAM unified delta: {delta_u:.1f} MB')

---
## Seção 7 — Gate G3: Throughput GPU (target ≥ 3× bucketed)

In [ ]:
import time

def bench_strategy(model_name, strategy, n_pos=100, n_freqs=1, reps=10):
    """Mede throughput (mod/h) para um modelo + estratégia após warmup."""
    _clear_all_caches()
    m = get_canonical_model(model_name)
    z = np.linspace(m.min_depth - 2, m.max_depth + 2, n_pos)
    freqs = np.array([20000.0 * (i + 1) for i in range(n_freqs)])

    ctx = build_static_context(
        m.rho_h, m.rho_v, m.esp, z,
        freqs_hz=freqs,
        tr_spacing_m=1.0, dip_deg=0.0,
        strategy=strategy,
    )

    # Warmup (compila JIT)
    H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()

    # Medição pós-warmup
    t0 = time.perf_counter()
    for _ in range(reps):
        H = forward_pure_jax(ctx.rho_h_jnp, ctx.rho_v_jnp, ctx)
    H.block_until_ready()
    elapsed = time.perf_counter() - t0

    ms_per_call = elapsed * 1e3 / reps
    modh = 3600.0 / (elapsed / reps)  # modelos/hora
    return ms_per_call, modh


print('Gate G3: Benchmark throughput unified vs bucketed...')
print(f'Dispositivo: {jax.default_backend().upper()}')
print(f'Configuração: oklahoma_3/5/28 × 100 posições × 1 freq × 10 reps\n')

bench_results = []
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    ms_b, modh_b = bench_strategy(model_name, 'bucketed', n_pos=100, reps=10)
    ms_u, modh_u = bench_strategy(model_name, 'unified',  n_pos=100, reps=10)
    ratio = modh_u / modh_b  # speedup unified vs bucketed (> 1 = unified mais rápido)
    xla_b = results_xla[(model_name, 'bucketed')]
    xla_u = results_xla[(model_name, 'unified')]
    bench_results.append({
        'model': model_name,
        'ms_b': ms_b, 'modh_b': modh_b,
        'ms_u': ms_u, 'modh_u': modh_u,
        'ratio': ratio,
        'xla_b': xla_b, 'xla_u': xla_u,
    })

print(f'{'Modelo':15s} {'XLA_b':>6s} {'XLA_u':>6s} '
      f'{'bucketed_ms':>12s} {'unified_ms':>11s} '
      f'{'speedup':>8s} {'Status':>8s}')
print('-' * 80)
for r in bench_results:
    status = '✅' if r['ratio'] >= 1.0 else ('⚠️' if r['ratio'] >= 0.5 else '❌')
    print(f'{r["model"]:15s} {r["xla_b"]:>6d} {r["xla_u"]:>6d} '
          f'{r["ms_b"]:>10.2f}ms {r["ms_u"]:>9.2f}ms '
          f'{r["ratio"]:>7.2f}× {status}')

# Gate G3: speedup oklahoma_28 >= 3× em GPU (diferente do CPU onde era 1.75×)
ratio_28 = next(r['ratio'] for r in bench_results if r['model'] == 'oklahoma_28')
G3_PASS = ratio_28 >= (3.0 if IS_GPU_BACKEND else 0.7)  # Gate CPU é mais folgado
gate_label = '≥3× (GPU)' if IS_GPU_BACKEND else '≥0.7× (CPU soft gate)'
print(f'\nGate G3: Speedup unified oklahoma_28 {gate_label}: {"✅ PASS" if G3_PASS else "❌ FAIL"}')
print(f'  Speedup medido: {ratio_28:.2f}×')

---
## Seção 8 — Benchmark Estendido (Multi-freq e Multi-posição)

In [ ]:
print('Benchmark estendido: oklahoma_28 × configurações variadas...\n')

configs_ext = [
    {'n_pos': 100,  'n_freqs': 1,  'label': 'baseline (100 pos, 1 freq)'},
    {'n_pos': 600,  'n_freqs': 1,  'label': 'grande   (600 pos, 1 freq)'},
    {'n_pos': 100,  'n_freqs': 3,  'label': 'multi-f  (100 pos, 3 freqs)'},
    {'n_pos': 600,  'n_freqs': 3,  'label': 'produção (600 pos, 3 freqs)'},
]

print(f'{'Configuração':30s} {'bucketed_ms':>12s} {'unified_ms':>12s} {'speedup':>9s}')
print('-' * 70)

for cfg in configs_ext:
    ms_b, _ = bench_strategy('oklahoma_28', 'bucketed',
                              n_pos=cfg['n_pos'], n_freqs=cfg['n_freqs'], reps=5)
    ms_u, _ = bench_strategy('oklahoma_28', 'unified',
                              n_pos=cfg['n_pos'], n_freqs=cfg['n_freqs'], reps=5)
    ratio = ms_b / ms_u  # speedup unified vs bucketed
    print(f'{cfg["label"]:30s} {ms_b:>10.2f}ms {ms_u:>10.2f}ms {ratio:>8.2f}×')

print('\n(speedup > 1 significa unified mais rápido que bucketed)')

---
## Seção 9 — Backward Compatibility (default = bucketed)

In [ ]:
print('Verificando backward compatibility: build_static_context sem strategy...')

m_3 = get_canonical_model('oklahoma_3')
z_test = np.linspace(m_3.min_depth - 2, m_3.max_depth + 2, 10)

# Sem parâmetro strategy — deve usar default "bucketed"
ctx_default = build_static_context(
    m_3.rho_h, m_3.rho_v, m_3.esp, z_test,
    freqs_hz=np.array([20000.0]),
    tr_spacing_m=1.0, dip_deg=0.0,
    # strategy omitido intencionalmente
)

print(f'ctx.strategy sem argumento: "{ctx_default.strategy}"')
COMPAT_PASS = ctx_default.strategy == 'bucketed'
print(f'Default preservado como "bucketed": {"✅ PASS" if COMPAT_PASS else "❌ FAIL"}')

# Verifica que forward roda corretamente com default
_clear_all_caches()
H_default = forward_pure_jax(ctx_default.rho_h_jnp, ctx_default.rho_v_jnp, ctx_default)
H_default.block_until_ready()
print(f'Forward com default "bucketed": ✅ OK (shape={H_default.shape})')

---
## Seção 10 — Relatório Final GO/NO-GO

In [ ]:
print('═' * 70)
print('  RELATÓRIO GO/NO-GO — JAX Unified JIT v1.5.0 (Sprint 10)')
print('═' * 70)
print(f'  Dispositivo  : {jax.default_backend().upper()} ({GPU_INFO if HAS_GPU else "CPU"})')
print(f'  JAX versão   : {jax.__version__}')
print(f'  Pacote       : geosteering_ai.simulation v{SIM_VERSION}')
print('─' * 70)

# Coleta resultados Sprint 10
gates = {
    'G1 — XLA count == 1 (oklahoma_28 unified)': G1_PASS,
    'G4 — Paridade <1e-10 (3 modelos canônicos)': G4_PASS,
    'G5 — jacfwd sem NaN/Inf (alta-ρ unified)':   G5_PASS,
    'Backward compat — default bucketed preservado': COMPAT_PASS,
}

# Gates GPU
if G2_PASS is None:
    gates['G2 — VRAM unified < 1 GB'] = None
else:
    gates['G2 — VRAM unified < 1 GB (GPU gate)'] = G2_PASS
gates[f'G3 — Speedup unified oklahoma_28 (GPU gate)'] = G3_PASS

print()
all_pass = True
for label, result in gates.items():
    if result is None:
        status = '⚠️  N/A (sem GPU)'
    elif result:
        status = '✅ PASS'
    else:
        status = '❌ FAIL'
        all_pass = False
    print(f'  {label:<52s} {status}')

print()
print('─' * 70)

if all_pass and IS_GPU_BACKEND:
    print('  🟢  SPRINT 10: GO — Todos os gates aprovados em GPU')
    print('  →  Baseline 5.46× (100pos×1f) confirmado')
elif all_pass and not IS_GPU_BACKEND:
    print('  🟡  SPRINT 10: PARCIAL — Gates CPU aprovados; pendente validação GPU')
    print('  →  AÇÃO: Re-executar em runtime GPU T4')
else:
    print('  🔴  SPRINT 10: NO-GO — Um ou mais gates falharam')
    failed = [k for k, v in gates.items() if v is False]
    for f in failed:
        print(f'     • Falhou: {f}')

print('═' * 70)
print()
print('  ➡️  Continuar para Seção 11 → Sprint 12: Diagnóstico Regressão + vmap_real')

In [ ]:
# ── Tabela de paridade detalhada ──────────────────────────────────────────────
print('Paridade detalhada unified vs bucketed:')
print(f'{'Modelo':15s} {'max|H_b - H_u|':>18s} {'4 ordens < gate':>17s}')
print('-' * 55)
for model_name, max_abs in results_parity.items():
    ordens = -np.log10(max_abs) - (-np.log10(1e-10))
    print(f'{model_name:15s} {max_abs:>18.3e} {ordens:>+15.1f} ordens')

print()
print('XLA consolidação:')
print(f'{'Modelo':15s} {'bucketed':>10s} {'unified':>10s} {'consolidação':>14s}')
print('-' * 55)
for model_name in ['oklahoma_3', 'oklahoma_5', 'oklahoma_28']:
    xla_b = results_xla[(model_name, 'bucketed')]
    xla_u = results_xla[(model_name, 'unified')]
    print(f'{model_name:15s} {xla_b:>10d} {xla_u:>10d} {xla_b}× → 1')

In [ ]:
# ── Limpeza final de caches ───────────────────────────────────────────────────
_clear_all_caches()
print('Caches JAX limpos.')
print('Notebook de validação concluído. ✅')

---
## Seção 11 — Sprint 12: Diagnóstico de Regressão 600×3 Freqs (E2/E11)

Roda a matriz 4-ponto de benchmark focada na regressão Sprint 12:

| Configuração | Hipótese | Gate |
|:-------------|:---------|:----:|
| 100pos × 1f (baseline) | Sprint 10 replicado | ≥ 3× |
| 600pos × 1f (n_pos grande) | Escalonamento linear OK | ≥ 1.5× |
| 100pos × 3f (multi-freq) | Fusão kernel OK | ≥ 1.5× |
| **600pos × 3f (produção)** | **Hipótese: L2 cache pressure** | **≥ 1.5×** |

Gera plots comparativos e define `reg_detected` (bool) para uso nas Seções 12–14.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

print('Sprint 12 — Diagnóstico Regressão: oklahoma_28 × 4 configs × bucketed vs unified')
print(f'Backend: {jax.default_backend().upper()}\n')

# ── Matrix de benchmark Sprint 12 ─────────────────────────────────────────────
# Foco nas 4 configs que revelam a regressão 600×3:
S12_MATRIX = [
    (100, 1, 'baseline   (100pos, 1f)'),
    (600, 1, 'grande     (600pos, 1f)'),
    (100, 3, 'multi-freq (100pos, 3f)'),
    (600, 3, 'produção   (600pos, 3f)'),
]

sprint12_bench_results = []
print(f'{"Configuração":28s} {"bucketed":>10s} {"unified":>10s} {"speedup":>9s} {"Status"}')
print('-' * 72)

for n_pos, nf, label in S12_MATRIX:
    ms_b, _ = bench_strategy('oklahoma_28', 'bucketed', n_pos=n_pos, n_freqs=nf, reps=5)
    ms_u, _ = bench_strategy('oklahoma_28', 'unified',  n_pos=n_pos, n_freqs=nf, reps=5)
    ratio = ms_b / ms_u

    sprint12_bench_results.append({
        'label': label, 'n_pos': n_pos, 'nf': nf,
        'bucketed_ms': ms_b, 'unified_ms': ms_u, 'ratio': ratio,
    })

    status = '✅ OK' if ratio >= 1.5 else ('⚠️  PARCIAL' if ratio >= 1.0 else '🔴 REGRESSÃO')
    print(f'{label:28s} {ms_b:>8.2f}ms {ms_u:>8.2f}ms {ratio:>8.2f}× {status}')

# ── Diagnóstico ───────────────────────────────────────────────────────────────
reg_detected = any(r['nf'] == 3 and r['n_pos'] == 600 and r['ratio'] < 1.5
                   for r in sprint12_bench_results)

print()
if reg_detected:
    print('⚠️  Regressão detectada em 600pos × 3f (ratio < 1.5×):')
    for r in sprint12_bench_results:
        if r['ratio'] < 1.5:
            print(f'   • {r["label"]}: {r["ratio"]:.2f}×')
    print('\nHipótese H1: pressão L2 cache — tensores (600, 3, 28, 201) float64 ≈ 6.7 MB > L2 T4 (4 MB)')
    print('→ Executar XLA profiling (Seção 13) para confirmação')
else:
    print('✅ Sem regressão — todos os configs com speedup ≥ 1.5×')

# ── Plots ──────────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f'Sprint 12 — Diagnóstico Regressão (oklahoma_28, {jax.default_backend().upper()})',
             fontsize=13, fontweight='bold')

labels_short = [r['label'].split('(')[1].replace(')', '') for r in sprint12_bench_results]
ms_b_arr = [r['bucketed_ms'] for r in sprint12_bench_results]
ms_u_arr = [r['unified_ms']  for r in sprint12_bench_results]
ratios    = [r['ratio']       for r in sprint12_bench_results]
x = np.arange(len(labels_short))
w = 0.35

# Latência
ax1.bar(x - w/2, ms_b_arr, w, label='bucketed', color='#E8A838', alpha=0.85)
ax1.bar(x + w/2, ms_u_arr, w, label='unified',  color='#4C6EE8', alpha=0.85)
ax1.set_xticks(x)
ax1.set_xticklabels(labels_short, rotation=15, ha='right', fontsize=9)
ax1.set_ylabel('Latência (ms)')
ax1.set_title('Latência por configuração')
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# Speedup
colors = ['#2ECC71' if r >= 1.5 else '#E74C3C' for r in ratios]
bars = ax2.bar(x, ratios, color=colors, alpha=0.85)
ax2.axhline(y=1.5, color='crimson', linestyle='--', linewidth=1.5, label='Gate 1.5×')
ax2.axhline(y=1.0, color='gray',   linestyle=':',  linewidth=1.0, label='break-even')
ax2.set_xticks(x)
ax2.set_xticklabels(labels_short, rotation=15, ha='right', fontsize=9)
ax2.set_ylabel('Speedup (bucketed / unified)')
ax2.set_title('Speedup unified (verde=PASS, vermelho=REGRESSÃO)')
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')

# Anotações de valor
for bar, ratio in zip(bars, ratios):
    ax2.annotate(f'{ratio:.2f}×', xy=(bar.get_x() + bar.get_width() / 2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig('sprint12_regression_diag.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Gráfico salvo: sprint12_regression_diag.png')

# VRAM no momento
vram_now = _get_gpu_vram_mb()
if vram_now:
    print(f'\nVRAM atual: {vram_now:.0f} MB')

---
## Seção 12 — Sprint 12: find_layers_tr_jax + vmap_real (Gates G6/G7/G8)

Valida os três gates Sprint 12 novos:

| Gate | Teste | Target |
|:-----|:------|:------:|
| **G6** | `find_layers_tr_jax` sweep 800+ pontos + boundary exata vs Numba | diff == 0 (int) |
| **G7** | `vmap_real` vs Python loop — max paridade `\|H_loop - H_vmap\|` | < 1e-12 |
| **G8** | `vmap_real` speedup em GPU (100pos × 3f × 3TR × 5ang) | ≥ 1.5× |

In [ ]:
import time as _time

from geosteering_ai.simulation._jax import find_layers_tr_jax, find_layers_tr_jax_vmap
from geosteering_ai.simulation._numba.geometry import find_layers_tr, sanitize_profile
from geosteering_ai.simulation._jax.multi_forward import simulate_multi_jax
from geosteering_ai.simulation.config import SimulationConfig

# ── Gate G6: Paridade bit-exata find_layers_tr_jax vs Numba ───────────────────
print('Gate G6: find_layers_tr_jax paridade vs Numba (diff == 0 sweep)...')

m_28 = get_canonical_model('oklahoma_28')
prof_np = sanitize_profile(m_28.esp)
prof_jnp = jnp.asarray(prof_np, dtype=jnp.float64)
n_layers = len(m_28.rho_h)

# Sweep: contínuo + exatamente nas interfaces + logo acima/abaixo
rng_g6 = np.random.default_rng(42)
depths_cont = np.linspace(m_28.min_depth - 5, m_28.max_depth + 5, 800)
depths_bdry = np.concatenate([
    prof_np[1:-1],            # exatamente na interface
    prof_np[1:-1] - 1e-9,     # logo acima
    prof_np[1:-1] + 1e-9,     # logo abaixo
])
all_depths = np.concatenate([depths_cont, depths_bdry])

h0_sweep = all_depths
z_sweep = rng_g6.choice(all_depths, size=len(all_depths), replace=False)

# Numba reference
ct_numba = np.array([find_layers_tr(h0, z, prof_np, n_layers)[0]
                     for h0, z in zip(h0_sweep, z_sweep)])
cr_numba = np.array([find_layers_tr(h0, z, prof_np, n_layers)[1]
                     for h0, z in zip(h0_sweep, z_sweep)])

# JAX
ct_jax, cr_jax = find_layers_tr_jax_vmap(
    jnp.asarray(h0_sweep), jnp.asarray(z_sweep), prof_jnp, n_layers
)
ct_jax = np.array(ct_jax)
cr_jax = np.array(cr_jax)

diff_t = int(np.max(np.abs(ct_jax - ct_numba)))
diff_r = int(np.max(np.abs(cr_jax - cr_numba)))

print(f'  Pontos testados : {len(h0_sweep)} ({len(depths_cont)} contínuo + {len(depths_bdry)} boundary)')
print(f'  Max diff TX (camad_t): {diff_t}')
print(f'  Max diff RX (camad_r): {diff_r}')

G6_PASS = diff_t == 0 and diff_r == 0
print(f'\nGate G6: find_layers_tr_jax diff==0: {"✅ PASS" if G6_PASS else "❌ FAIL"}')

# ── Gate G7: vmap_real paridade vs Python loop ─────────────────────────────────
print('\nGate G7: vmap_real paridade vs Python loop (oklahoma_28, 3TR × 5ang)...')

z_100 = np.linspace(m_28.min_depth - 2, m_28.max_depth + 2, 100)
freqs_3f = np.array([20000.0, 40000.0, 80000.0])
tr_spacings_3 = np.array([0.5, 1.0, 1.5])
dip_degs_5 = np.array([0.0, 15.0, 30.0, 45.0, 60.0])

cfg_loop = SimulationConfig(jax_strategy='unified', jax_vmap_real=False)
cfg_vmap = SimulationConfig(jax_strategy='unified', jax_vmap_real=True)

_clear_all_caches()
res_loop = simulate_multi_jax(
    m_28.rho_h, m_28.rho_v, m_28.esp,
    z_100, freqs_3f, tr_spacings_3, dip_degs_5, cfg=cfg_loop
)
_clear_all_caches()
res_vmap = simulate_multi_jax(
    m_28.rho_h, m_28.rho_v, m_28.esp,
    z_100, freqs_3f, tr_spacings_3, dip_degs_5, cfg=cfg_vmap
)

# Extrair tensor — tenta H_tensor, fallback para array
def _get_tensor(res):
    if hasattr(res, 'H_tensor'):
        return jnp.asarray(np.array(res.H_tensor))
    if hasattr(res, 'H_array'):
        return jnp.asarray(np.array(res.H_array))
    return jnp.asarray(np.array(res))

H_loop = _get_tensor(res_loop)
H_vmap = _get_tensor(res_vmap)
parity_g7 = float(jnp.abs(H_loop - H_vmap).max())

G7_PASS = parity_g7 < 1e-12
print(f'  Max |H_loop - H_vmap|: {parity_g7:.3e}')
print(f'\nGate G7: vmap_real paridade < 1e-12: {"✅ PASS" if G7_PASS else "❌ FAIL"}')

# ── Gate G8: vmap_real speedup em GPU ─────────────────────────────────────────
print('\nGate G8: vmap_real speedup vs Python loop (100pos × 3f × 3TR × 5ang)...')
_reps = 5

# Warmup (2 rodadas cada)
for _ in range(2):
    simulate_multi_jax(m_28.rho_h, m_28.rho_v, m_28.esp, z_100, freqs_3f, tr_spacings_3, dip_degs_5, cfg=cfg_loop)
    simulate_multi_jax(m_28.rho_h, m_28.rho_v, m_28.esp, z_100, freqs_3f, tr_spacings_3, dip_degs_5, cfg=cfg_vmap)

t0 = _time.perf_counter()
for _ in range(_reps):
    simulate_multi_jax(m_28.rho_h, m_28.rho_v, m_28.esp, z_100, freqs_3f, tr_spacings_3, dip_degs_5, cfg=cfg_loop)
ms_loop = (_time.perf_counter() - t0) * 1e3 / _reps

t0 = _time.perf_counter()
for _ in range(_reps):
    simulate_multi_jax(m_28.rho_h, m_28.rho_v, m_28.esp, z_100, freqs_3f, tr_spacings_3, dip_degs_5, cfg=cfg_vmap)
ms_vmap_g8 = (_time.perf_counter() - t0) * 1e3 / _reps

speedup_g8 = ms_loop / ms_vmap_g8
G8_PASS = speedup_g8 >= 1.5

print(f'  Python loop : {ms_loop:.1f} ms')
print(f'  vmap_real   : {ms_vmap_g8:.1f} ms')
print(f'  Speedup     : {speedup_g8:.2f}×')
print(f'\nGate G8: vmap_real ≥ 1.5×: {"✅ PASS" if G8_PASS else f"⚠️  {speedup_g8:.2f}× (abaixo, opt-in preservado)"}')

---
## Seção 13 — Sprint 12: XLA Profiling (Condicional — ativa se S11 mostrar regressão)

Se `reg_detected = True` (regressão em 600×3 freqs), esta seção:
1. Coleta um XLA trace via `jax.profiler.trace`
2. Mede overhead supralinear para confirmar hipótese L2 cache (H1)
3. Salva trace em `/tmp/xla_sprint12_600x3` para análise TensorBoard

Se `reg_detected = False`, exibe mensagem de skip e define `G9_PASS = True`.

In [ ]:
# ── XLA Profiling condicional — só executa se S11 detectar regressão ──────────
print('Seção 13 — XLA Profiling para diagnóstico de regressão...\n')

if not reg_detected:
    print('✅ Sem regressão em 600×3 freqs — XLA profiling não necessário.')
    print('   Continue para Seção 14 (Relatório Final).')
    G9_PASS = True
else:
    ratio_600x3 = next(
        (r['ratio'] for r in sprint12_bench_results if r['nf'] == 3 and r['n_pos'] == 600),
        None
    )
    print(f'⚠️  Regressão detectada: 600×3 ratio={ratio_600x3:.2f}× (< 1.5× gate)')
    print('   Coletando XLA trace...\n')

    # ── Construir contexto 600pos × 3f unified ───────────────────────────────
    _clear_all_caches()
    m_28 = get_canonical_model('oklahoma_28')
    z_600 = np.linspace(m_28.min_depth - 2, m_28.max_depth + 2, 600)
    freqs_3 = np.logspace(4, np.log10(1e5), 3)
    ctx_prof = build_static_context(
        m_28.rho_h, m_28.rho_v, m_28.esp, z_600,
        freqs_hz=freqs_3, tr_spacing_m=1.0, dip_deg=0.0,
        strategy='unified',
    )
    # Warmup
    H = forward_pure_jax(ctx_prof.rho_h_jnp, ctx_prof.rho_v_jnp, ctx_prof)
    H.block_until_ready()

    # ── Profiler trace ────────────────────────────────────────────────────────
    import jax.profiler as _jprofiler
    profile_dir = '/tmp/xla_sprint12_600x3'
    os.makedirs(profile_dir, exist_ok=True)

    with _jprofiler.trace(profile_dir, create_perfetto_link=False):
        H = forward_pure_jax(ctx_prof.rho_h_jnp, ctx_prof.rho_v_jnp, ctx_prof)
        H.block_until_ready()

    print(f'✅ XLA trace salvo em: {profile_dir}')
    print('\n   Para visualizar:')
    print('   1. Download do arquivo .xplane.pb no Colab Files (ícone pasta à esquerda)')
    print('   2. tensorboard --logdir=/tmp/xla_sprint12_600x3')
    print('   3. Aba "Trace Viewer" → procurar kernels lentos')

    # ── Análise overhead supralinear ─────────────────────────────────────────
    print('\n   Análise: escalonamento de latência 100×1 → 600×3...')
    ms_100x1_u, _ = bench_strategy('oklahoma_28', 'unified', n_pos=100, n_freqs=1, reps=5)
    ms_600x3_u, _ = bench_strategy('oklahoma_28', 'unified', n_pos=600, n_freqs=3, reps=5)
    ms_100x1_b, _ = bench_strategy('oklahoma_28', 'bucketed', n_pos=100, n_freqs=1, reps=5)
    ms_600x3_b, _ = bench_strategy('oklahoma_28', 'bucketed', n_pos=600, n_freqs=3, reps=5)

    scale_factor = (600 * 3) / (100 * 1)  # = 18×
    overhead_u = (ms_600x3_u / ms_100x1_u) / scale_factor
    overhead_b = (ms_600x3_b / ms_100x1_b) / scale_factor

    print(f'   Fator de escala esperado (posições × freqs): {scale_factor:.0f}×')
    print(f'   Overhead supralinear unified : {overhead_u:.2f}× (>1 = pior que linear)')
    print(f'   Overhead supralinear bucketed: {overhead_b:.2f}×')

    if overhead_u > 1.5:
        print('\n   → Overhead supralinear confirma HIPÓTESE H1 (pressão L2 cache):')
        print('     Tensores (600, 3, 28, 201) float64 ≈ 6.7 MB > L2 T4 (4 MB)')
        print('     Recomendação: aplicar E4 (position chunking) em v1.6.1')
    elif overhead_u > 1.1:
        print('\n   → Overhead moderado — hipótese H2 (materialização excessiva vmap)')
        print('     Recomendação: análise HLO dump via XLA_FLAGS=--xla_dump_to=...')
    else:
        print('\n   → Overhead linear — regressão pode ser ruído de medição')

    G9_PASS = True  # Diagnóstico executado com sucesso

print(f'\nGate G9 — Diagnóstico regressão 600×3: {"✅ PASS" if G9_PASS else "⏳"}')

---
## Seção 14 — Sprint 12: Relatório Final E11 (GO/NO-GO v1.6.1)

In [ ]:
from datetime import datetime

print('═' * 70)
print('  RELATÓRIO E11 — Sprint 12 v1.6.0 → v1.6.1 GO/NO-GO')
print('═' * 70)
print(f'  Data         : {datetime.now().strftime("%Y-%m-%d %H:%M")} (UTC-3)')
print(f'  Dispositivo  : {jax.default_backend().upper()} ({GPU_INFO if HAS_GPU else "CPU"})')
print(f'  JAX versão   : {jax.__version__}')
print(f'  Pacote       : geosteering_ai.simulation v{SIM_VERSION}')
print('─' * 70)

# ── Sprint 10 gates (Seção 10) ────────────────────────────────────────────────
s10_gates = {
    'G1 — XLA count == 1 (oklahoma_28 unified)': G1_PASS,
    'G2 — VRAM unified < 1 GB':                  G2_PASS,
    'G3 — Speedup unified ≥ 3× (GPU gate)':      G3_PASS,
    'G4 — Paridade <1e-10 (3 modelos)':           G4_PASS,
    'G5 — jacfwd sem NaN/Inf (alta-ρ)':           G5_PASS,
}

# ── Sprint 12 gates (Seções 11-13) ────────────────────────────────────────────
s12_gates = {
    'G6 — find_layers_tr_jax diff==0 vs Numba':          G6_PASS,
    'G7 — vmap_real paridade < 1e-12 vs Python loop':    G7_PASS,
    f'G8 — vmap_real speedup ({speedup_g8:.2f}× ≥ 1.5×)': G8_PASS,
    'G9 — Regressão 600×3 diagnosticada':                G9_PASS,
}

def _fmt(result):
    if result is None: return '⚠️  N/A'
    return '✅ PASS' if result else '❌ FAIL'

print('\n  SPRINT 10 (baseline v1.5.0):')
s10_all = True
for lbl, r in s10_gates.items():
    if r is False: s10_all = False
    print(f'    {lbl:<52s} {_fmt(r)}')

print('\n  SPRINT 12 (v1.6.0 — E11):')
s12_all = True
for lbl, r in s12_gates.items():
    if r is False: s12_all = False
    print(f'    {lbl:<52s} {_fmt(r)}')

print('\n─' * 70)

# Regressão resumo
print('\n  DIAGNÓSTICO REGRESSÃO (oklahoma_28):')
print(f'  {"Configuração":30s} {"bucketed":>10s} {"unified":>10s} {"speedup":>9s} {"Status":>8s}')
print('  ' + '-' * 72)
for r in sprint12_bench_results:
    status = '✅' if r['ratio'] >= 1.5 else ('⚠️ ' if r['ratio'] >= 1.0 else '🔴')
    print(f'  {r["label"]:30s} {r["bucketed_ms"]:>8.1f}ms {r["unified_ms"]:>8.1f}ms {r["ratio"]:>8.2f}× {status}')

# ── Decisão v1.6.1 ────────────────────────────────────────────────────────────
print('\n─' * 70)
all_pass_e11 = s10_all and s12_all and (G2_PASS is not False)

if all_pass_e11:
    print('  🟢  DECISÃO v1.6.1: GO — Todos os gates Sprint 10 + Sprint 12 aprovados')
    print()
    print('  AÇÕES RECOMENDADAS v1.6.1:')
    if speedup_g8 >= 2.0:
        print('  → Flip jax_vmap_real default False → True (speedup ≥ 2×)')
    else:
        print(f'  → Manter jax_vmap_real=False default (speedup={speedup_g8:.2f}×, abaixo de 2×)')
    print('  → NÃO flip jax_strategy default "bucketed" (defer v1.7.0 após soak)')
    n_reg = sum(1 for r in sprint12_bench_results if r['ratio'] < 1.5)
    if n_reg == 0:
        print('  → Sem regressão 600×3: E4-E6 otimizações NÃO urgentes em v1.6.1')
    else:
        print(f'  → {n_reg} config(s) com regressão: avaliar E4 (position chunking) em v1.6.1')
    print()
    print('  → Abrir PR #26 (v1.6.1) com apenas as ações acima confirmadas')
else:
    print('  🔴  DECISÃO: NO-GO — Um ou mais gates falharam')
    for gates_dict in [s10_gates, s12_gates]:
        for lbl, r in gates_dict.items():
            if r is False:
                print(f'  → Investigar: {lbl}')

print('═' * 70)

# Salvar relatório texto
lines = [
    f'Sprint 12 E11 GPU Report',
    f'Data: {datetime.now().strftime("%Y-%m-%d %H:%M")}',
    f'Hardware: {GPU_INFO if HAS_GPU else "CPU"}',
    f'JAX: {jax.__version__}',
    f'Pacote: geosteering_ai.simulation v{SIM_VERSION}',
    '',
    'Sprint 10 Gates:',
]
for lbl, r in s10_gates.items():
    lines.append(f'  {lbl}: {"PASS" if r else ("N/A" if r is None else "FAIL")}')
lines += ['', 'Sprint 12 Gates (E11):']
for lbl, r in s12_gates.items():
    lines.append(f'  {lbl}: {"PASS" if r else "FAIL"}')
lines += ['', 'Benchmark Regressão (oklahoma_28):']
for r in sprint12_bench_results:
    lines.append(
        f'  {r["label"]}: B={r["bucketed_ms"]:.1f}ms U={r["unified_ms"]:.1f}ms ratio={r["ratio"]:.2f}x'
    )

with open('sprint12_e11_report.txt', 'w') as f:
    f.write('\n'.join(lines))
print('\n✅ Relatório salvo: sprint12_e11_report.txt')